In [ ]:
"""03_prepare_poi_data.py

功能：
1. 读取上海市 POI 数据；
2. 筛选杨浦区 POI；
3. 将高德 GCJ-02 坐标转换为 WGS84；
4. 校验源分类；
5. 生成规划语义分类 category_planning；
6. 生成老年友好、15分钟生活圈、步行友好等场景标签；
7. 去重；
8. 输出 CSV / Parquet / GeoJSON；
9. 生成数据质量报告。

输入：
- D:/POI 数据/2024年华东地区/上海市/上海市POI数据.csv
- 可选：POI源分类_2024.xlsx
- 可选：杨浦区边界 GeoJSON

输出：
- D:/AI/urban_renewal_data/processed/poi/2024/
"""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point


# =========================
# 1. 路径配置
# =========================

INPUT_POI_CSV = Path(r"D:\POI 数据\2024年华东地区\上海市\上海市POI数据.csv")

# 你的分类表。如果你已经把附件表格复制到本地，请修改为实际路径。
SOURCE_CATEGORY_XLSX = Path(r"D:\POI 数据\POI源分类_2024.xlsx")

# 如果有杨浦区边界文件，填入路径；如果没有，就保持 None。
# 边界文件通常应为 WGS84 坐标系。
# YANGPU_BOUNDARY_PATH: Path | None = None
# 示例：
YANGPU_BOUNDARY_PATH = Path(r"D:\街景\杨浦区_县.geojson")

OUTPUT_DIR = Path(r"D:\AI\Dataset-urban plan\POI data")

OUTPUT_CSV = OUTPUT_DIR / "poi_yangpu_clean.csv"
OUTPUT_PARQUET = OUTPUT_DIR / "poi_yangpu_clean.parquet"
OUTPUT_GEOJSON = OUTPUT_DIR / "poi_yangpu_clean.geojson"
OUTPUT_CATEGORY_MAPPING = OUTPUT_DIR / "poi_category_mapping_generated.csv"
OUTPUT_UNKNOWN_CATEGORY = OUTPUT_DIR / "poi_unknown_source_categories.csv"
OUTPUT_REPORT = OUTPUT_DIR / "poi_quality_report.md"

TARGET_CITY = "上海市"
TARGET_DISTRICT = "杨浦区"

SOURCE_CRS = "GCJ02"
ANALYSIS_CRS = "WGS84"


# =========================
# 2. 坐标转换：GCJ-02 -> WGS84
# =========================

PI = math.pi
A = 6378245.0
EE = 0.00669342162296594323


def out_of_china(lon: float, lat: float) -> bool:
    return not (73.66 < lon < 135.05 and 3.86 < lat < 53.55)


def transform_lat(lon: float, lat: float) -> float:
    ret = (
        -100.0
        + 2.0 * lon
        + 3.0 * lat
        + 0.2 * lat * lat
        + 0.1 * lon * lat
        + 0.2 * math.sqrt(abs(lon))
    )
    ret += (
        (20.0 * math.sin(6.0 * lon * PI) + 20.0 * math.sin(2.0 * lon * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (20.0 * math.sin(lat * PI) + 40.0 * math.sin(lat / 3.0 * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (160.0 * math.sin(lat / 12.0 * PI) + 320.0 * math.sin(lat * PI / 30.0))
        * 2.0
        / 3.0
    )
    return ret


def transform_lon(lon: float, lat: float) -> float:
    ret = (
        300.0
        + lon
        + 2.0 * lat
        + 0.1 * lon * lon
        + 0.1 * lon * lat
        + 0.1 * math.sqrt(abs(lon))
    )
    ret += (
        (20.0 * math.sin(6.0 * lon * PI) + 20.0 * math.sin(2.0 * lon * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (20.0 * math.sin(lon * PI) + 40.0 * math.sin(lon / 3.0 * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (150.0 * math.sin(lon / 12.0 * PI) + 300.0 * math.sin(lon / 30.0 * PI))
        * 2.0
        / 3.0
    )
    return ret


def gcj02_to_wgs84(gcj_lon: float, gcj_lat: float) -> tuple[float, float]:
    """
    高德 GCJ-02 坐标转 WGS84。
    """
    if out_of_china(gcj_lon, gcj_lat):
        return gcj_lon, gcj_lat

    dlat = transform_lat(gcj_lon - 105.0, gcj_lat - 35.0)
    dlon = transform_lon(gcj_lon - 105.0, gcj_lat - 35.0)

    radlat = gcj_lat / 180.0 * PI
    magic = math.sin(radlat)
    magic = 1 - EE * magic * magic
    sqrt_magic = math.sqrt(magic)

    dlat = (dlat * 180.0) / ((A * (1 - EE)) / (magic * sqrt_magic) * PI)
    dlon = (dlon * 180.0) / (A / sqrt_magic * math.cos(radlat) * PI)

    mg_lat = gcj_lat + dlat
    mg_lon = gcj_lon + dlon

    wgs_lon = gcj_lon * 2 - mg_lon
    wgs_lat = gcj_lat * 2 - mg_lat

    return wgs_lon, wgs_lat


# =========================
# 3. 读取工具函数
# =========================

def read_csv_safely(path: Path) -> pd.DataFrame:
    """
    尝试用多种中文常见编码读取 CSV。
    """
    if not path.exists():
        raise FileNotFoundError(f"POI CSV 文件不存在：{path}")

    encodings = ["utf-8-sig", "utf-8", "gb18030", "gbk"]

    last_error: Exception | None = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"无法读取 CSV：{path}，最后错误：{last_error}")


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def load_source_category_pairs(path: Path) -> set[tuple[str, str]]:
    """
    读取源分类表，返回合法的 大类/中类 组合。
    如果文件不存在，则返回空集合。
    """
    if not path.exists():
        print(f"未找到源分类表，跳过分类校验：{path}")
        return set()

    df = pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]

    if "大类" not in df.columns or "中类" not in df.columns:
        raise ValueError("源分类表必须包含字段：大类, 中类")

    pairs = set()

    for _, row in df.iterrows():
        big = normalize_text(row["大类"])
        mid = normalize_text(row["中类"])
        if big and mid:
            pairs.add((big, mid))

    return pairs


# =========================
# 4. POI 规划语义分类映射
# =========================

SPECIFIC_CATEGORY_MAPPING: dict[tuple[str, str], str] = {
    # 交通设施
    ("交通设施", "公交站"): "公交站",
    ("交通设施", "地铁"): "地铁站",
    ("交通设施", "停车场"): "停车设施",
    ("交通设施", "火车"): "对外交通设施",
    ("交通设施", "长途汽车"): "对外交通设施",
    ("交通设施", "飞机"): "对外交通设施",
    ("交通设施", "港口码头"): "对外交通设施",
    ("交通设施", "服务区"): "交通服务设施",

    # 医疗保健
    ("医疗保健", "综合医院"): "医疗",
    ("医疗保健", "专科医院"): "医疗",
    ("医疗保健", "诊所"): "医疗",
    ("医疗保健", "急救中心"): "医疗",
    ("医疗保健", "疾病预防"): "医疗",
    ("医疗保健", "医药销售"): "药店",
    ("医疗保健", "动物医疗"): "宠物医疗",

    # 商务住宅
    ("商务住宅", "住宅区"): "居住社区",
    ("商务住宅", "宿舍"): "居住社区",
    ("商务住宅", "别墅区"): "居住社区",
    ("商务住宅", "商住两用楼宇"): "居住社区",
    ("商务住宅", "社区中心"): "社区服务设施",
    ("商务住宅", "产业园"): "产业办公",
    ("商务住宅", "写字楼"): "产业办公",
    ("商务住宅", "工业大楼"): "产业办公",

    # 旅游景点 / 公共空间
    ("旅游景点", "公园"): "公园绿地",
    ("旅游景点", "广场"): "公共空间",
    ("旅游景点", "植物园"): "公园绿地",
    ("旅游景点", "动物园"): "公园绿地",
    ("旅游景点", "纪念馆"): "文化设施",
    ("旅游景点", "红色旅游"): "文化设施",
    ("旅游景点", "景点"): "文化旅游",
    ("旅游景点", "宗教"): "文化旅游",
    ("旅游景点", "水族馆"): "文化旅游",

    # 生活服务
    ("生活服务", "公厕"): "公厕",
    ("生活服务", "邮局"): "生活服务",
    ("生活服务", "电讯营业厅"): "生活服务",
    ("生活服务", "公共事业"): "生活服务",
    ("生活服务", "洗衣"): "生活服务",
    ("生活服务", "美容理发"): "生活服务",
    ("生活服务", "洗浴推拿"): "生活服务",
    ("生活服务", "物流"): "物流服务",

    # 科教文化
    ("科教文化", "幼儿园"): "教育",
    ("科教文化", "小学"): "教育",
    ("科教文化", "中学"): "教育",
    ("科教文化", "高等教育"): "教育",
    ("科教文化", "职业技术教育"): "教育",
    ("科教文化", "成人教育"): "教育",
    ("科教文化", "培训单位"): "教育培训",
    ("科教文化", "图书馆"): "文化设施",
    ("科教文化", "博物馆"): "文化设施",
    ("科教文化", "科技馆"): "文化设施",
    ("科教文化", "文化宫"): "文化设施",
    ("科教文化", "美术展览"): "文化设施",
    ("科教文化", "档案馆"): "文化设施",
    ("科教文化", "会展展览"): "文化设施",
    ("科教文化", "科研单位"): "科研教育",

    # 购物消费
    ("购物消费", "便利店"): "超市便利",
    ("购物消费", "超市"): "超市便利",
    ("购物消费", "市场"): "菜场",
    ("购物消费", "购物中心"): "商业设施",
    ("购物消费", "百货商场"): "商业设施",
    ("购物消费", "商业街"): "商业设施",
    ("购物消费", "家居建材"): "商业设施",
    ("购物消费", "家电数码"): "商业设施",
    ("购物消费", "文体用品"): "商业设施",
    ("购物消费", "花鸟鱼虫"): "商业设施",

    # 运动健身
    ("运动健身", "综合体育馆"): "文体设施",
    ("运动健身", "健身中心"): "文体设施",
    ("运动健身", "户外健身场所"): "文体设施",
    ("运动健身", "篮球"): "文体设施",
    ("运动健身", "足球"): "文体设施",
    ("运动健身", "乒乓球"): "文体设施",
    ("运动健身", "羽毛球"): "文体设施",
    ("运动健身", "游泳"): "文体设施",
    ("运动健身", "网球"): "文体设施",

    # 休闲娱乐
    ("休闲娱乐", "度假养老"): "养老服务",
    ("休闲娱乐", "剧场"): "文化设施",
    ("休闲娱乐", "电影院"): "文化设施",
    ("休闲娱乐", "棋牌室"): "休闲娱乐",
    ("休闲娱乐", "KTV"): "休闲娱乐",
    ("休闲娱乐", "酒吧"): "休闲娱乐",
    ("休闲娱乐", "网吧"): "休闲娱乐",
    ("休闲娱乐", "游乐场"): "休闲娱乐",

    # 汽车相关
    ("汽车相关", "加油站"): "能源补给设施",
    ("汽车相关", "充电站"): "能源补给设施",
    ("汽车相关", "加气站"): "能源补给设施",
    ("汽车相关", "停车场"): "停车设施",

    # 其他
    ("酒店住宿", "旅馆"): "酒店住宿",
    ("酒店住宿", "经济型连锁酒店"): "酒店住宿",
    ("酒店住宿", "青旅"): "酒店住宿",
    ("金融机构", "银行"): "金融服务",
    ("金融机构", "ATM"): "金融服务",
    ("金融机构", "保险"): "金融服务",
    ("餐饮美食", "中国菜"): "餐饮",
    ("餐饮美食", "小吃快餐"): "餐饮",
    ("餐饮美食", "咖啡"): "餐饮",
    ("餐饮美食", "茶座"): "餐饮",
    ("餐饮美食", "蛋糕甜品店"): "餐饮",
    ("餐饮美食", "外国菜"): "餐饮",
}


BIG_CATEGORY_FALLBACK: dict[str, str] = {
    "交通设施": "交通设施",
    "医疗保健": "医疗",
    "商务住宅": "居住与办公",
    "旅游景点": "文化旅游",
    "生活服务": "生活服务",
    "科教文化": "教育文化",
    "购物消费": "商业服务",
    "运动健身": "文体设施",
    "休闲娱乐": "休闲娱乐",
    "公司企业": "产业办公",
    "汽车相关": "汽车服务",
    "酒店住宿": "酒店住宿",
    "金融机构": "金融服务",
    "餐饮美食": "餐饮",
}


def map_planning_category(category_big: str, category_mid: str) -> str:
    big = normalize_text(category_big)
    mid = normalize_text(category_mid)

    if (big, mid) in SPECIFIC_CATEGORY_MAPPING:
        return SPECIFIC_CATEGORY_MAPPING[(big, mid)]

    if big in BIG_CATEGORY_FALLBACK:
        return BIG_CATEGORY_FALLBACK[big]

    return "其他"


def make_scenario_tags(category_planning: str) -> str:
    """
    生成 Agent 场景标签。
    用 | 分隔，便于 CSV 保存。
    """
    elderly_relevant = {
        "医疗",
        "药店",
        "养老服务",
        "公交站",
        "地铁站",
        "菜场",
        "超市便利",
        "公园绿地",
        "公共空间",
        "公厕",
        "社区服务设施",
        "文体设施",
        "文化设施",
        "生活服务",
    }

    life_circle_relevant = {
        "医疗",
        "药店",
        "养老服务",
        "教育",
        "教育培训",
        "菜场",
        "超市便利",
        "公园绿地",
        "公共空间",
        "公厕",
        "公交站",
        "地铁站",
        "社区服务设施",
        "文体设施",
        "文化设施",
        "生活服务",
        "商业设施",
        "餐饮",
    }

    walkability_relevant = {
        "公交站",
        "地铁站",
        "公园绿地",
        "公共空间",
        "公厕",
        "菜场",
        "超市便利",
        "医疗",
        "社区服务设施",
        "文体设施",
        "文化设施",
    }

    tags = []

    if category_planning in elderly_relevant:
        tags.append("elderly_friendly")

    if category_planning in life_circle_relevant:
        tags.append("life_circle")

    if category_planning in walkability_relevant:
        tags.append("walkability")

    return "|".join(tags)


def make_priority_weight(category_planning: str) -> float:
    """
    给不同 POI 类型一个简单权重，便于后续排序或筛选。
    这不是最终规划评分，只是数据层面的初步权重。
    """
    high = {
        "医疗",
        "药店",
        "养老服务",
        "公交站",
        "地铁站",
        "菜场",
        "公园绿地",
        "公厕",
        "社区服务设施",
    }

    medium = {
        "超市便利",
        "教育",
        "文体设施",
        "文化设施",
        "公共空间",
        "生活服务",
        "商业设施",
    }

    if category_planning in high:
        return 1.0

    if category_planning in medium:
        return 0.7

    return 0.4


# =========================
# 5. 主流程
# =========================

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("读取 POI 原始数据...")
    raw_df = read_csv_safely(INPUT_POI_CSV)
    raw_df.columns = [str(c).strip() for c in raw_df.columns]

    required_cols = ["名称", "大类", "中类", "经度", "纬度", "省份", "地级市", "区县"]
    missing_cols = [c for c in required_cols if c not in raw_df.columns]
    if missing_cols:
        raise ValueError(f"POI CSV 缺少必要字段：{missing_cols}")

    raw_count = len(raw_df)

    print(f"原始 POI 数量：{raw_count}")

    print("标准化字段...")
    df = raw_df.rename(
        columns={
            "名称": "name",
            "大类": "category_big",
            "中类": "category_mid",
            "经度": "lon_gcj02",
            "纬度": "lat_gcj02",
            "省份": "province",
            "地级市": "city",
            "区县": "district",
        }
    ).copy()

    text_cols = [
        "name",
        "category_big",
        "category_mid",
        "province",
        "city",
        "district",
    ]

    for col in text_cols:
        df[col] = df[col].apply(normalize_text)

    df["lon_gcj02"] = pd.to_numeric(df["lon_gcj02"], errors="coerce")
    df["lat_gcj02"] = pd.to_numeric(df["lat_gcj02"], errors="coerce")

    print("删除无名称或无坐标记录...")
    invalid_name_count = (df["name"] == "").sum()
    invalid_coord_count = df[["lon_gcj02", "lat_gcj02"]].isna().any(axis=1).sum()

    df = df[(df["name"] != "") & df["lon_gcj02"].notna() & df["lat_gcj02"].notna()].copy()

    print("检查上海范围内坐标...")
    shanghai_coord_mask = (
        df["lon_gcj02"].between(120.5, 122.5)
        & df["lat_gcj02"].between(30.5, 32.0)
    )

    out_of_range_count = (~shanghai_coord_mask).sum()
    df = df[shanghai_coord_mask].copy()

    print("筛选上海市记录...")
    city_mask = df["city"].str.contains("上海", na=False)
    not_shanghai_city_count = (~city_mask).sum()
    df = df[city_mask].copy()

    print("转换坐标：GCJ-02 -> WGS84...")
    converted = df.apply(
        lambda row: gcj02_to_wgs84(row["lon_gcj02"], row["lat_gcj02"]),
        axis=1,
    )

    df["lon_wgs84"] = [item[0] for item in converted]
    df["lat_wgs84"] = [item[1] for item in converted]

    print("生成空间点...")
    geometry = [
        Point(lon, lat)
        for lon, lat in zip(df["lon_wgs84"], df["lat_wgs84"])
    ]

    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

    print("筛选杨浦区 POI...")

    gdf["in_yangpu_by_attr"] = gdf["district"].str.contains("杨浦", na=False)

    if YANGPU_BOUNDARY_PATH is not None and YANGPU_BOUNDARY_PATH.exists():
        print(f"使用边界文件进行空间筛选：{YANGPU_BOUNDARY_PATH}")

        boundary_gdf = gpd.read_file(YANGPU_BOUNDARY_PATH)

        if boundary_gdf.empty:
            raise ValueError("杨浦区边界文件为空。")

        if boundary_gdf.crs is None:
            boundary_gdf = boundary_gdf.set_crs(epsg=4326)
        else:
            boundary_gdf = boundary_gdf.to_crs(epsg=4326)

        boundary_gdf = boundary_gdf[["geometry"]].copy()
        boundary_gdf["boundary_id"] = 1

        joined = gpd.sjoin(
            gdf,
            boundary_gdf,
            how="left",
            predicate="intersects",
        )

        joined["in_yangpu_by_boundary"] = joined["boundary_id"].notna()
        drop_cols = [c for c in ["index_right", "boundary_id"] if c in joined.columns]
        joined = joined.drop(columns=drop_cols)

        # 如果有边界，以空间筛选为准。
        yangpu_gdf = joined[joined["in_yangpu_by_boundary"]].copy()
        filter_method = "boundary_spatial_filter"

    else:
        print("未使用边界文件，仅根据区县字段筛选杨浦区。")
        gdf["in_yangpu_by_boundary"] = False
        yangpu_gdf = gdf[gdf["in_yangpu_by_attr"]].copy()
        filter_method = "district_attribute_filter"

    yangpu_before_dedupe_count = len(yangpu_gdf)

    print(f"杨浦区 POI 数量，去重前：{yangpu_before_dedupe_count}")

    print("校验源分类...")
    source_pairs = load_source_category_pairs(SOURCE_CATEGORY_XLSX)

    if source_pairs:
        yangpu_gdf["source_category_valid"] = yangpu_gdf.apply(
            lambda row: (row["category_big"], row["category_mid"]) in source_pairs,
            axis=1,
        )
    else:
        yangpu_gdf["source_category_valid"] = True

    print("生成规划语义分类...")
    yangpu_gdf["category_planning"] = yangpu_gdf.apply(
        lambda row: map_planning_category(row["category_big"], row["category_mid"]),
        axis=1,
    )

    yangpu_gdf["category_source_path"] = (
        yangpu_gdf["category_big"] + "/" + yangpu_gdf["category_mid"]
    )

    yangpu_gdf["scenario_tags"] = yangpu_gdf["category_planning"].apply(make_scenario_tags)

    yangpu_gdf["is_elderly_relevant"] = yangpu_gdf["scenario_tags"].str.contains(
        "elderly_friendly",
        na=False,
    )
    yangpu_gdf["is_life_circle_relevant"] = yangpu_gdf["scenario_tags"].str.contains(
        "life_circle",
        na=False,
    )
    yangpu_gdf["is_walkability_relevant"] = yangpu_gdf["scenario_tags"].str.contains(
        "walkability",
        na=False,
    )

    yangpu_gdf["poi_priority_weight"] = yangpu_gdf["category_planning"].apply(
        make_priority_weight
    )

    print("生成标准 POI ID...")
    yangpu_gdf = yangpu_gdf.reset_index(drop=True)
    yangpu_gdf["poi_id"] = [
        f"yp_poi_2024_{i + 1:06d}" for i in range(len(yangpu_gdf))
    ]

    print("去重...")
    yangpu_gdf["name_norm"] = yangpu_gdf["name"].str.replace(r"\s+", "", regex=True)
    yangpu_gdf["lon_round6"] = yangpu_gdf["lon_gcj02"].round(6)
    yangpu_gdf["lat_round6"] = yangpu_gdf["lat_gcj02"].round(6)

    duplicate_mask = yangpu_gdf.duplicated(
        subset=[
            "name_norm",
            "category_big",
            "category_mid",
            "lon_round6",
            "lat_round6",
        ],
        keep="first",
    )

    duplicate_count = int(duplicate_mask.sum())

    yangpu_gdf = yangpu_gdf[~duplicate_mask].copy()

    print(f"去重后 POI 数量：{len(yangpu_gdf)}")

    print("生成检索文本字段...")
    yangpu_gdf["search_text"] = yangpu_gdf.apply(
        lambda row: " ".join(
            [
                row["name"],
                row["category_big"],
                row["category_mid"],
                row["category_planning"],
                row["district"],
                row["city"],
            ]
        ),
        axis=1,
    )

    yangpu_gdf["source"] = "amap_poi_2024"
    yangpu_gdf["source_crs"] = SOURCE_CRS
    yangpu_gdf["analysis_crs"] = ANALYSIS_CRS
    yangpu_gdf["data_year"] = 2024

    print("整理输出字段...")

    output_cols = [
        "poi_id",
        "name",
        "category_big",
        "category_mid",
        "category_source_path",
        "category_planning",
        "scenario_tags",
        "is_elderly_relevant",
        "is_life_circle_relevant",
        "is_walkability_relevant",
        "poi_priority_weight",
        "lon_gcj02",
        "lat_gcj02",
        "lon_wgs84",
        "lat_wgs84",
        "province",
        "city",
        "district",
        "in_yangpu_by_attr",
        "in_yangpu_by_boundary",
        "source_category_valid",
        "source",
        "source_crs",
        "analysis_crs",
        "data_year",
        "search_text",
        "geometry",
    ]

    existing_cols = [c for c in output_cols if c in yangpu_gdf.columns]
    yangpu_gdf = yangpu_gdf[existing_cols].copy()

    print("保存 CSV...")
    csv_df = pd.DataFrame(yangpu_gdf.drop(columns="geometry"))
    csv_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("保存 Parquet...")
    yangpu_gdf.to_parquet(OUTPUT_PARQUET, index=False)

    print("保存 GeoJSON...")
    yangpu_gdf.to_file(OUTPUT_GEOJSON, driver="GeoJSON", encoding="utf-8")

    print("保存分类映射表...")
    mapping_rows = []

    unique_pairs = (
        yangpu_gdf[["category_big", "category_mid", "category_planning"]]
        .drop_duplicates()
        .sort_values(["category_big", "category_mid"])
    )

    for _, row in unique_pairs.iterrows():
        mapping_rows.append(
            {
                "category_big": row["category_big"],
                "category_mid": row["category_mid"],
                "category_planning": row["category_planning"],
            }
        )

    mapping_df = pd.DataFrame(mapping_rows)
    mapping_df.to_csv(OUTPUT_CATEGORY_MAPPING, index=False, encoding="utf-8-sig")

    print("保存未知源分类...")
    unknown_df = (
        yangpu_gdf[~yangpu_gdf["source_category_valid"]][
            ["category_big", "category_mid", "category_source_path"]
        ]
        .drop_duplicates()
        .sort_values(["category_big", "category_mid"])
    )
    unknown_df.to_csv(OUTPUT_UNKNOWN_CATEGORY, index=False, encoding="utf-8-sig")

    print("生成质量报告...")

    category_big_stats = (
        yangpu_gdf["category_big"].value_counts().to_dict()
    )
    category_planning_stats = (
        yangpu_gdf["category_planning"].value_counts().to_dict()
    )
    scenario_stats = {
        "elderly_friendly": int(yangpu_gdf["is_elderly_relevant"].sum()),
        "life_circle": int(yangpu_gdf["is_life_circle_relevant"].sum()),
        "walkability": int(yangpu_gdf["is_walkability_relevant"].sum()),
    }

    invalid_source_category_count = int((~yangpu_gdf["source_category_valid"]).sum())

    report = f"""# 杨浦区 POI 数据预处理质量报告

## 1. 输入数据

- 原始 POI CSV：`{INPUT_POI_CSV}`
- 源分类表：`{SOURCE_CATEGORY_XLSX if SOURCE_CATEGORY_XLSX.exists() else "未使用"}`
- 杨浦区边界：`{YANGPU_BOUNDARY_PATH if YANGPU_BOUNDARY_PATH else "未使用"}`
- 输出目录：`{OUTPUT_DIR}`

## 2. 处理方法

- 原始坐标系：高德 GCJ-02
- 分析坐标系：WGS84
- 杨浦区筛选方法：`{filter_method}`
- 目标城市：`{TARGET_CITY}`
- 目标区县：`{TARGET_DISTRICT}`

## 3. 数据数量

| 指标 | 数量 |
|---|---:|
| 原始 POI 数量 | {raw_count} |
| 无名称记录数 | {invalid_name_count} |
| 无效坐标记录数 | {invalid_coord_count} |
| 上海范围外坐标记录数 | {out_of_range_count} |
| 非上海市记录数 | {not_shanghai_city_count} |
| 杨浦区 POI 数量，去重前 | {yangpu_before_dedupe_count} |
| 重复记录数 | {duplicate_count} |
| 杨浦区 POI 数量，去重后 | {len(yangpu_gdf)} |
| 源分类异常记录数 | {invalid_source_category_count} |

## 4. 原始大类分布

```json
{json.dumps(category_big_stats, ensure_ascii=False, indent=2)}
{json.dumps(category_planning_stats, ensure_ascii=False, indent=2)}
{json.dumps(scenario_stats, ensure_ascii=False, indent=2)}
"""

    with open(OUTPUT_REPORT, "w", encoding="utf-8") as f:
        f.write(report)
    print("处理完成。")
    print(f"CSV: {OUTPUT_CSV}")
    print(f"Parquet: {OUTPUT_PARQUET}")
    print(f"GeoJSON: {OUTPUT_GEOJSON}")
    print(f"Report: {OUTPUT_REPORT}")

if __name__ == "__main__":
    main()